<a href="https://colab.research.google.com/github/zzpsah/UDISE/blob/main/UDISE%2B_Autmation_Suite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 UDISE+ Professional Automation Suite

### Made with ❤️ by Eternal Student

---

If this tool saves your time and effort,
consider supporting with Tea ☕ or Coffee ☕

**UPI:** `eternalstudent@cnrb`

---

## 📌 Workflow

1. Setup Environment
2. Authentication
3. Detect School
4. Fetch Students
5. General Profile Module
6. Upload & Validation
7. Submit Updates
8. Download Logs



In [1]:
#@title ⚙️ Setup Environment { display-mode: "form" }

!pip install -q requests pandas openpyxl tqdm

import requests as _req

GAS_URL = "https://script.google.com/macros/s/AKfycbx9Lo1C72nJ89VlxBNkv9wuCl7Uw6pNBTha9PaIJ-LhvgLzq0oCYH7W84_ODcyzwQ7n/exec"

try:
    _r = _req.get(GAS_URL, params={"action": "stats"}, timeout=10)
    _count = _r.json().get("unique_users", "—") if _r.status_code == 200 else "—"
except Exception:
    _count = "—"

print("✅ Environment Ready")
print(f"👥 Total schools using this tool: {_count}")

✅ Environment Ready
👥 Total schools using this tool: 19


In [ ]:
#@title 🔐 Authentication { display-mode: "form" }
from getpass import getpass
import re
import requests
import pandas as pd
import io
import os
import json
import time

cookie_string = getpass("Paste full UDISE+ Cookie: ")

jsession_match = re.search(r'JSESSIONID=([^;]+)', cookie_string)
xsrf_match = re.search(r'XSRF-TOKEN=([^;]+)', cookie_string)

if not jsession_match or not xsrf_match:
    raise ValueError("❌ Could not extract session tokens")

JSESSIONID = jsession_match.group(1)
XSRF_TOKEN = xsrf_match.group(1)

BASE_URL = "https://sdms.udiseplus.gov.in"

session = requests.Session()

session.cookies.set("JSESSIONID", JSESSIONID)
session.cookies.set("XSRF-TOKEN", XSRF_TOKEN)

HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Origin": BASE_URL,
    "Referer": f"{BASE_URL}/g0/",
    "User-Agent": "Mozilla/5.0",
    "X-XSRF-TOKEN": XSRF_TOKEN,
}

resp = session.get(f"{BASE_URL}/p0/check-session", headers=HEADERS)

if resp.status_code == 200:
    print("✅ Session Valid")
else:
    raise RuntimeError("❌ Session Invalid")


In [ ]:
#@title 🏫 Detect School { display-mode: "form" }
current_url = input("Paste UDISE+ School URL: ").strip()

school_match = re.search(r'/school/(\d+)/', current_url)

if not school_match:
    raise ValueError("❌ Invalid school URL")

SCHOOL_ID = school_match.group(1)
print(f"✅ School ID Detected: {SCHOOL_ID}")

# Get username silently
_USERNAME = ""
try:
    import subprocess
    _result = subprocess.run(
        ["gcloud", "config", "get-value", "account"],
        capture_output=True, text=True, timeout=5
    )
    _USERNAME = _result.stdout.strip()
except Exception:
    pass

# Ping — Timestamp + Username + School ID
try:
    _req.get(GAS_URL, params={
        "action"        : "ping",
        "school_id"     : SCHOOL_ID,
        "student_count" : "",
        "username"      : _USERNAME,
        "ip"            : "",
        "city"          : "",
        "session_id"    : "",
    }, timeout=10)
except Exception:
    pass

Paste UDISE+ School URL: https://sdms.udiseplus.gov.in/g0/#/school/2502758/listAllStudentCy
✅ School ID Detected: 2502758


In [ ]:
#@title 👨‍🎓 Fetch Current Academic session Students { display-mode: "form" }
resp = session.get(
    f"{BASE_URL}/p0/api/cy/students/all/{SCHOOL_ID}",
    headers=HEADERS
)

if resp.status_code != 200:
    raise RuntimeError("❌ Failed to fetch students")

students = resp.json().get("data", [])

pen_to_studentid = {}

for s in students:
    pen = str(s.get("studentCodeNat", "")).strip()
    sid = s.get("studentId")

    if pen and sid:
        pen_to_studentid[pen] = sid

print(f"✅ Students fetched successfully: {len(students)}")


✅ Students fetched successfully: 244


In [ ]:
#@title 👨‍🎓 Download ALL STUDENTS DETAILS Excel (Name, Mother, Father, PEN, DOB, Mobile, Masked Aadhaar, APAAR ID) { display-mode: "form" }
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import re
import time

def mask_aadhaar_standalone(aadhaar_val):
    if not aadhaar_val:
        return "N/A"
    raw = re.sub(r'\D', '', str(aadhaar_val))
    if len(raw) >= 12:
        return f"XXXX-XXXX-{raw[-4:]}"
    elif len(raw) >= 4:
        return f"XXXX-XXXX-{raw[-4:]}"
    return str(aadhaar_val) if str(aadhaar_val).strip() else "N/A"

def get_apaar_id_standalone(s):
    for key in ["apaarId", "apaarNo", "aparId", "studentApaarId", "apaar_id", "apaarNumber"]:
        val = s.get(key)
        if val is not None and str(val).strip() not in ("", "0", "None", "null"):
            return str(val).strip()
    return "N/A"

def generate_all_students_excel(output_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "ALL STUDENTS DETAILS"
    ws.row_dimensions[1].height = 32

    ALL_COLUMNS = [
        "S. No.", "Class", "Section", "Student Name",
        "Mother Name", "Father Name", "PEN Number",
        "Date of Birth", "Mobile No.", "Masked Aadhaar", "APAAR ID"
    ]
    ALL_COL_WIDTHS = [8, 10, 10, 26, 24, 24, 18, 16, 16, 20, 22]

    FILL_HDR = PatternFill("solid", fgColor="1F4E79")
    FONT_HDR = Font(bold=True, color="FFFFFF", size=10)
    FONT_NORM = Font(color="000000", size=10)
    ALIGN_CTR = Alignment(horizontal="center", vertical="center")
    ALIGN_LEFT = Alignment(horizontal="left", vertical="center")
    thin = Side(style="thin", color="CCCCCC")
    BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)

    for ci, hdr in enumerate(ALL_COLUMNS, 1):
        c = ws.cell(1, ci, hdr)
        c.font = FONT_HDR
        c.fill = FILL_HDR
        c.alignment = ALIGN_CTR
        c.border = BORDER

    for ci, w in enumerate(ALL_COL_WIDTHS, 1):
        ws.column_dimensions[get_column_letter(ci)].width = w

    ws.freeze_panes = "G2"

    from IPython.display import display as _disp, HTML as _HTML
    _prog = _HTML(f"""
    <div style='background:#161b22;border:1px solid #1f6feb;border-radius:8px;padding:12px 18px;font-family:Inter,sans-serif;margin:4px 0;'>
      <div style='color:#c9d1d9;font-size:0.85rem;margin-bottom:8px;'>
        Fetching All Students Details (<strong style='color:#C9A84C'>{len(pen_to_studentid)}</strong> students) ...
      </div>
      <div style='background:#0d1117;border-radius:4px;height:6px;overflow:hidden;'>
        <div style='height:6px;width:0%;background:linear-gradient(90deg,#C9A84C,#e8c96d);'></div>
      </div>
    </div>""")
    _disp(_prog, display_id='all_students_progress')

    rows_done = 0
    for pen, sid in pen_to_studentid.items():
        gresp = session.get(f"{BASE_URL}/p0/api/cy/students/{sid}", headers=HEADERS)
        if gresp.status_code != 200:
            continue
        s = gresp.json().get("data", {})
        row = rows_done + 2

        ws.cell(row, 1, rows_done + 1).alignment = ALIGN_CTR
        ws.cell(row, 2, s.get("classDesc", "")).alignment = ALIGN_CTR
        ws.cell(row, 3, s.get("sectionDesc", "")).alignment = ALIGN_CTR
        ws.cell(row, 4, s.get("studentName", "")).alignment = ALIGN_LEFT
        ws.cell(row, 5, s.get("motherName", "")).alignment = ALIGN_LEFT
        ws.cell(row, 6, s.get("fatherName", "")).alignment = ALIGN_LEFT
        ws.cell(row, 7, s.get("studentCodeNat", "") or pen).alignment = ALIGN_CTR
        ws.cell(row, 8, s.get("dob", "")).alignment = ALIGN_CTR
        ws.cell(row, 9, s.get("primaryMobile", "") or "N/A").alignment = ALIGN_CTR
        ws.cell(row, 10, mask_aadhaar_standalone(s.get("uuid"))).alignment = ALIGN_CTR
        ws.cell(row, 11, get_apaar_id_standalone(s)).alignment = ALIGN_CTR

        for ci in range(1, 12):
            c = ws.cell(row, ci)
            c.border = BORDER
            c.font = FONT_NORM

        rows_done += 1
        if rows_done % 5 == 0 or rows_done == len(pen_to_studentid):
            _pct = int(rows_done / len(pen_to_studentid) * 100)
            from IPython.display import update_display
            update_display(_HTML(f"""
            <div style='background:#161b22;border:1px solid #1f6feb;border-radius:8px;padding:12px 18px;font-family:Inter,sans-serif;margin:4px 0;'>
              <div style='color:#c9d1d9;font-size:0.85rem;margin-bottom:8px;'>
                Fetching All Students Details (<strong style='color:#C9A84C'>{len(pen_to_studentid)}</strong> students) ...
              </div>
              <div style='background:#0d1117;border-radius:4px;height:6px;overflow:hidden;'>
                <div style='height:6px;width:{_pct}%;background:linear-gradient(90deg,#C9A84C,#e8c96d);'></div>
              </div>
              <div style='color:#8b949e;font-size:0.75rem;margin-top:6px;'>{rows_done} / {len(pen_to_studentid)} done ({_pct}%)</div>
            </div>"""), display_id='all_students_progress')
        time.sleep(0.05)

    footer_cell = ws.cell(rows_done + 3, 1, f'Generated by Eternal Student | eternalstudent@cnrb | School: {SCHOOL_ID}')
    footer_cell.font = Font(color='C9A84C', italic=True, size=9)

    wb.save(output_path)
    from google.colab import files
    files.download(output_path)

    from IPython.display import update_display
    update_display(_HTML(f"""
    <div style='background:#161b22;border:1px solid #238636;border-radius:8px;padding:14px 20px;font-family:Inter,sans-serif;'>
      <div style='display:flex;align-items:center;gap:10px;margin-bottom:10px;'>
        <span style='color:#3fb950;font-size:1.3rem;'>&#10003;</span>
        <span style='color:#c9d1d9;font-weight:600;'>ALL STUDENTS DETAILS Excel Exported Successfully</span>
      </div>
      <div style='color:#8b949e;font-size:0.82rem;'>
        Total <span style='color:#C9A84C;font-weight:600;'>{rows_done}</span> students written with Student Name, Mother Name, Father Name, PEN, DOB, Mobile No., Masked Aadhaar, and APAAR ID.
      </div>
    </div>"""), display_id='all_students_progress')

out_all = f"UDISE_All_Students_Details_{SCHOOL_ID}.xlsx"
generate_all_students_excel(out_all)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# GENERAL PROFILE UPDATE ( GP UPDATE ON UDISE+)

In [ ]:
#@title REFERENCE DATA  --  Excel label  <->  API code { display-mode: "form" }
BLOOD_GROUP = {
    "A+"  : "1",  "A-"  : "2",
    "B+"  : "3",  "B-"  : "4",
    "O+"  : "5",  "O-"  : "6",
    "AB+" : "7",  "AB-" : "8",
    "Under Investigation - Result will be updated soon": "9",
    "Unknown / Not Available": "0",
}

SOCIAL_CATEGORY = {
    "1 - General": 1,
    "2 - SC"     : 2,
    "3 - ST"     : 3,
    "4 - OBC"    : 4,
}

MINORITY_GROUP = {
    "1 - Muslim"     : 1,
    "2 - Christian"  : 2,
    "3 - Sikh"       : 3,
    "4 - Buddhist"   : 4,
    "5 - Jain"       : 5,
    "6 - Zoroastrian": 6,
    "7 - NA"         : 7,
}

YES_NO = {"Yes": 1, "No": 2}

DISABILITY_CERTI = {"Yes": "1", "No": "2", "NA": "9"}

OOSC_MAINSTREAMED = {"Yes": "1", "No": "2", "NA": "9"}

IMPAIRMENT_TYPE = {
    "1 - Blindness"                        : 1,
    "2 - Low Vision"                       : 2,
    "3 - Leprosy Cured"                    : 3,
    "4 - Hearing Impairment"               : 4,
    "5 - Locomotor Disability"             : 5,
    "6 - Mental Retardation"               : 6,
    "7 - Mental Illness"                   : 7,
    "8 - Autism Spectrum Disorder"         : 8,
    "9 - Cerebral Palsy"                   : 9,
    "10 - Specific Learning Disabilities"  : 10,
    "11 - Speech and Language Disability"  : 11,
    "12 - Thalassemia"                     : 12,
    "13 - Haemophilia"                     : 13,
    "14 - Sickle Cell Disease"             : 14,
    "15 - Multiple Disabilities"           : 15,
    "16 - Acid Attack Victim"              : 16,
    "17 - Parkinsons Disease"              : 17,
    "18 - Dwarfism"                        : 18,
    "19 - Muscular Dystrophy"              : 19,
    "20 - Chronic Neurological conditions" : 20,
    "21 - Multiple Sclerosis"              : 21,
}

MOTHER_TONGUE = {
    "1 - ASSAMESE - Assamese"           : 1,
    "2 - ASSAMESE - Bodo"               : 2,
    "3 - ASSAMESE - Deuri"              : 3,
    "4 - ASSAMESE - Dimasa"             : 4,
    "5 - ASSAMESE - Karbi"              : 5,
    "6 - ASSAMESE - Mising"             : 6,
    "7 - ASSAMESE - Rabha"              : 7,
    "8 - ASSAMESE - Tiwa"               : 8,
    "9 - BENGALI - Bengali"             : 9,
    "10 - BENGALI - Bishnupriya"        : 10,
    "11 - BENGALI - Chakma"             : 11,
    "12 - BENGALI - Hajong"             : 12,
    "13 - BENGALI - Munda"              : 13,
    "14 - BENGALI - Santali"            : 14,
    "15 - BENGALI - Tripuri"            : 15,
    "16 - DOGRI - Dogri"                : 16,
    "17 - GUJARATI - Gujarati"          : 17,
    "18 - GUJARATI - Bhili/Bhilodi"     : 18,
    "19 - GUJARATI - Gamit"             : 19,
    "20 - GUJARATI - Konkani"           : 20,
    "21 - GUJARATI - Vasavi"            : 21,
    "22 - HINDI - Bagheli/Baghel Khandi": 22,
    "23 - HINDI - Banjari"              : 23,
    "24 - HINDI - Bundeli/Bundel Khandi": 24,
    "25 - HINDI - Chhattisgarhi"        : 25,
    "26 - HINDI - Garhwali"             : 26,
    "27 - HINDI - Harauti"              : 27,
    "28 - HINDI - Bhojpuri"             : 28,
    "29 - HINDI - Kangri"               : 29,
    "30 - HINDI - Kumauni"              : 30,
    "31 - HINDI - Lamani/Lambadi"       : 31,
    "32 - HINDI - Magahi"               : 32,
    "33 - HINDI - Maithili"             : 33,
    "34 - HINDI - Malvi"                : 34,
    "35 - HINDI - Marwari"              : 35,
    "36 - HINDI - Mewari"               : 36,
    "37 - HINDI - Nimadi"               : 37,
    "38 - HINDI - Pahari"               : 38,
    "39 - HINDI - Rajasthani"           : 39,
    "40 - HINDI - Surgujia"             : 40,
    "41 - HINDI - Awadhi"               : 41,
    "42 - HINDI - Hindi"                : 42,
    "43 - KANNADA - Kannada"            : 43,
    "44 - KANNADA - Tulu"               : 44,
    "45 - KASHMIRI - Kashmiri"          : 45,
    "46 - KASHMIRI - Gojri"             : 46,
    "47 - KASHMIRI - Ladakhi"           : 47,
    "48 - MAITHILI - Maithili"          : 48,
    "49 - MALAYALAM - Malayalam"        : 49,
    "50 - MANIPURI - Manipuri"          : 50,
    "51 - MANIPURI - Tangkhul"          : 51,
    "52 - MARATHI - Marathi"            : 52,
    "53 - MARATHI - Gondi"              : 53,
    "54 - MARATHI - Halabi"             : 54,
    "55 - MARATHI - Kolami"             : 55,
    "56 - MARATHI - Korku"              : 56,
    "57 - MARATHI - Koya"               : 57,
    "58 - MARATHI - Varli"              : 58,
    "59 - NEPALI - Nepali"              : 59,
    "60 - NEPALI - Limbu"               : 60,
    "61 - ODIA - Odia"                  : 61,
    "62 - ODIA - Gondi"                 : 62,
    "63 - ODIA - Ho"                    : 63,
    "64 - ODIA - Juang"                 : 64,
    "65 - ODIA - Kharia"                : 65,
    "66 - ODIA - Kissan"                : 66,
    "67 - ODIA - Koya"                  : 67,
    "68 - ODIA - Munda"                 : 68,
    "69 - ODIA - Oraon/Kurukh"          : 69,
    "70 - ODIA - Santali"               : 70,
    "71 - PUNJABI - Punjabi"            : 71,
    "72 - PUNJABI - Dogri"              : 72,
    "73 - SANSKRIT - Sanskrit"          : 73,
    "74 - SANTHALI - Santali"           : 74,
    "75 - SINDHI - Sindhi"              : 75,
    "76 - TAMIL - Tamil"                : 76,
    "77 - TAMIL - Irula"                : 77,
    "78 - TAMIL - Kota"                 : 78,
    "79 - TAMIL - Toda"                 : 79,
    "80 - TELUGU - Telugu"              : 80,
    "81 - TELUGU - Gondi"               : 81,
    "82 - TELUGU - Koya"                : 82,
    "83 - URDU - Urdu"                  : 83,
    "84 - ENGLISH - English"            : 84,
    "85 - OTHERS - Others"              : 85,
    "86 - HINDI - Haryanvi"             : 86,
    "87 - HINDI - Mewati"               : 87,
    "88 - HINDI - Dhundhari"            : 88,
    "89 - HINDI - Ahirani"              : 89,
    "90 - HINDI - Wagdi"                : 90,
    "91 - TELUGU - Savara"              : 91,
    "92 - MANIPURI - Mao"               : 92,
    "93 - KONKANI - Konkani"            : 93,
    "94 - TAMIL - Badaga"               : 94,
    "95 - KANNADA - Kodava/Coorg"       : 95,
    "96 - BENGALI - Koch"               : 96,
    "97 - ODIA - Kisan"                 : 97,
    "98 - MARATHI - Katkari"            : 98,
    "99 - HINDI - Kurmali Thar"         : 99,
    "100 - HINDI - Sadri"               : 100,
    "101 - ODIA - Bhunjia"              : 101,
    "102 - HINDI - Brajbhasha"          : 102,
    "103 - HINDI - Bihari"              : 103,
    "104 - HINDI - Kanauji"             : 104,
    "105 - HINDI - Khortha/Khotta"      : 105,
    "106 - HINDI - Kurmali"             : 106,
    "107 - HINDI - Nagpuria"            : 107,
    "108 - HINDI - Surajpuri"           : 108,
    "109 - HINDI - Bagheli"             : 109,
    "110 - HINDI - Garo"                : 110,
    "111 - HINDI - Lohari"              : 111,
    "112 - HINDI - Shekhawati"          : 112,
    "113 - ASSAMESE - Rabha"            : 113,
    "114 - ASSAMESE - Khasi"            : 114,
    "115 - ASSAMESE - Garo"             : 115,
    "116 - ASSAMESE - Mizo/Lushai"      : 116,
    "117 - ASSAMESE - Hmar"             : 117,
    "118 - ASSAMESE - Kuki"             : 118,
    "119 - ASSAMESE - Ao"               : 119,
    "120 - ASSAMESE - Angami"           : 120,
    "121 - ASSAMESE - Lotha"            : 121,
    "122 - ASSAMESE - Sema"             : 122,
    "123 - ASSAMESE - Kabui"            : 123,
    "124 - ASSAMESE - Kom"              : 124,
    "125 - ASSAMESE - Paite"            : 125,
    "126 - ASSAMESE - Anal"             : 126,
    "127 - ASSAMESE - Thado"            : 127,
    "128 - ASSAMESE - Adi"              : 128,
    "129 - ASSAMESE - Nyishi"           : 129,
    "130 - ASSAMESE - Apatani"          : 130,
    "131 - ASSAMESE - Nocte"            : 131,
    "132 - ASSAMESE - Wancho"           : 132,
    "133 - ASSAMESE - Nissi/Dafla"      : 133,
    "134 - NEPALI - Gurung"             : 134,
    "135 - NEPALI - Rai"                : 135,
    "136 - NEPALI - Tamang"             : 136,
    "137 - NEPALI - Sherpa"             : 137,
    "138 - NEPALI - Sunwar"             : 138,
    "139 - NEPALI - Tharu"              : 139,
    "140 - NEPALI - Magar"              : 140,
    "141 - MARATHI - Pawri/Pavri"       : 141,
    "142 - TAMIL - Telugu"              : 142,
    "143 - TELUGU - Yerukala/Yerukula"  : 143,
    "144 - URDU - Urdu"                 : 144,
}

def _rev(d): return {v: k for k, v in d.items()}
BLOOD_GROUP_R       = _rev(BLOOD_GROUP)
SOCIAL_CATEGORY_R   = _rev(SOCIAL_CATEGORY)
MINORITY_GROUP_R    = _rev(MINORITY_GROUP)
YES_NO_R            = _rev(YES_NO)
DISABILITY_CERTI_R  = _rev(DISABILITY_CERTI)
OOSC_MAINSTREAMED_R = _rev(OOSC_MAINSTREAMED)
IMPAIRMENT_TYPE_R   = _rev(IMPAIRMENT_TYPE)
MOTHER_TONGUE_R     = _rev(MOTHER_TONGUE)

print(f"Reference data loaded.")
print(f"  Mother Tongue options : {len(MOTHER_TONGUE)}")
print(f"  Blood Group options   : {len(BLOOD_GROUP)}")
print(f"  Impairment types      : {len(IMPAIRMENT_TYPE)}")


In [ ]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


 **Generate Pre-filled Excel**

| Column range | Colour | Action |
|---|---|---|
| **A to K** | Grey (locked) | Demographic / identity &mdash; read-only reference, never updated |
| **L to AD** | White (editable) | Pre-filled with current values &mdash; edit and save |

Enum columns (Blood Group, Category, etc.) have **dropdown validation** &mdash; you can only select valid values. After editing, save the file and upload it in Step 8.

In [ ]:
#@title 📄 General Profile — Download Excel { display-mode: "form" }
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Protection, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter
from openpyxl.comments import Comment

COLUMNS = [
    ("Class",                "classDesc",          True,  None),
    ("Section",              "sectionDesc",         True,  None),
    ("DOB (DD/MM/YYYY)",     "dob",                 True,  None),
    ("PEN",                  "studentCodeNat",      True,  None),
    ("Student Name",         "studentName",         True,  None),
    ("Gender",               "__gender_label",      True,  None),
    ("Mother Name",          "motherName",          True,  None),
    ("Father Name",          "fatherName",          True,  None),
    ("Guardian Name",        "guardianName",        True,  None),
    ("Aadhaar No.",          "uuid",                True,  None),
    ("Name as per Aadhaar",  "nameAsUuid",          True,  None),
    ("Address",              "address",             False, None),
    ("Pincode",              "pincode",             False, None),
    ("Mobile No.",           "primaryMobile",       False, None),
    ("Alternate Mobile No.", "secondaryMobile",     False, None),
    ("Contact Email",        "email",               False, None),
    ("Mother Tongue",        "motherTongue",        False, "MT"),
    ("Social Category",      "socCatId",            False, "SC"),
    ("Minority Group",       "minorityId",          False, "MG"),
    ("BPL Beneficiary",      "isBplYN",             False, "YN"),
    ("AAY Beneficiary",      "aayBplYN",            False, "YN"),
    ("EWS / Disadvantaged",  "ewsYN",               False, "YN"),
    ("CWSN",                 "cwsnYN",              False, "YN"),
    ("Type of Impairment",   "impairmentType",      False, "IT"),
    ("Disability Certificate","disabilityCerti",    False, "DC"),
    ("Disability % (0-100)", "impairmentPercent",   False, None),
    ("Indian National",      "natIndYN",            False, "YN"),
    ("Out-of-School Child",  "ooscYN",              False, "YN"),
    ("When Mainstreamed",    "ooscMainstreamedYN",  False, "OM"),
    ("Blood Group",          "bloodGroup",          False, "BG"),
]

FILL_LOCKED  = PatternFill("solid", fgColor="D9D9D9")
FILL_HDR_LCK = PatternFill("solid", fgColor="404040")
FILL_HDR_UPD = PatternFill("solid", fgColor="1F4E79")
FILL_WHITE   = PatternFill("solid", fgColor="FFFFFF")
FONT_HDR     = Font(bold=True, color="FFFFFF", size=10)
FONT_LOCK    = Font(color="666666", size=10)
FONT_NORM    = Font(color="000000", size=10)
ALIGN_CTR    = Alignment(horizontal="center", vertical="center")
ALIGN_LEFT   = Alignment(horizontal="left",   vertical="center")
thin         = Side(style="thin", color="CCCCCC")
BORDER       = Border(left=thin, right=thin, top=thin, bottom=thin)
COL_WIDTHS   = [7,7,14,16,28,10,24,24,18,16,24,
                36,10,14,14,24,36,18,18,7,7,7,7,
                32,22,12,7,7,18,14]

def _api_to_label(api_field, dmap_key, s):
    """Convert raw API value to human-readable label for pre-filling."""
    if api_field == "__gender_label":
        return {1:"Male", 2:"Female", 3:"Transgender"}.get(s.get("gender"), "")
    val = s.get(api_field)
    if val is None or val == "": return ""
    if dmap_key == "SC": return SOCIAL_CATEGORY_R.get(val, str(val))
    if dmap_key == "MG": return MINORITY_GROUP_R.get(val,  str(val))
    if dmap_key == "YN": return YES_NO_R.get(val,           str(val))
    if dmap_key == "IT":
        # API may return a list for students with multiple impairments
        if isinstance(val, list):
            val = val[0] if val else None
            if val is None: return ""
        return IMPAIRMENT_TYPE_R.get(val, str(val))
    if dmap_key == "MT": return MOTHER_TONGUE_R.get(val,    str(val))
    if dmap_key == "BG": return BLOOD_GROUP_R.get(str(val), str(val))
    if dmap_key == "DC": return DISABILITY_CERTI_R.get(str(val), str(val))
    if dmap_key == "OM": return OOSC_MAINSTREAMED_R.get(str(val), str(val))
    if val == 0 and api_field not in ("pincode", "impairmentPercent"): return ""
    return str(val) if val is not None else ""

def generate_excel(output_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "GP Update"

    hl = wb.create_sheet("_Lists")
    hl.sheet_state = "hidden"
    mt_labels  = list(MOTHER_TONGUE.keys())
    imp_labels = list(IMPAIRMENT_TYPE.keys())
    for row_i, lbl in enumerate(mt_labels,  start=1): hl.cell(row_i, 1, lbl)
    for row_j, lbl in enumerate(imp_labels, start=1): hl.cell(row_j, 2, lbl)
    mt_ref  = f"_Lists!$A$1:$A${len(mt_labels)}"
    imp_ref = f"_Lists!$B$1:$B${len(imp_labels)}"

    ws.row_dimensions[1].height = 32
    for ci, (hdr, _, locked, _dmap) in enumerate(COLUMNS, 1):
        c = ws.cell(1, ci, hdr)
        c.font      = FONT_HDR
        c.fill      = FILL_HDR_LCK if locked else FILL_HDR_UPD
        c.alignment = ALIGN_CTR
        c.border    = BORDER

    ws["A1"].comment = Comment(
        "GREY columns (A-K): LOCKED - demographic data.\n"
        "BLUE columns (L-AD): EDITABLE - change these, then upload back.",
        "UDISE Script"
    )
    ws.freeze_panes = "E2"

    for ci, w in enumerate(COL_WIDTHS, 1):
        ws.column_dimensions[get_column_letter(ci)].width = w

    # Progress display
    from IPython.display import display as _disp, HTML as _HTML
    _prog = _HTML(f"""
    <div style='background:#161b22;border:1px solid #1f6feb;border-radius:8px;
                padding:12px 18px;font-family:Inter,sans-serif;margin:4px 0;'>
      <div style='color:#c9d1d9;font-size:0.85rem;margin-bottom:8px;'>
        Fetching <strong style='color:#C9A84C'>{len(pen_to_studentid)}</strong> student profiles ...
      </div>
      <div style='background:#0d1117;border-radius:4px;height:6px;overflow:hidden;'>
        <div id='es-bar' style='height:6px;width:0%;background:linear-gradient(90deg,#C9A84C,#e8c96d);transition:width 0.3s;'></div>
      </div>
      <div id='es-prog-txt' style='color:#8b949e;font-size:0.75rem;margin-top:6px;'>Starting ...</div>
    </div>""")
    _prog_handle = _disp(_prog, display_id='es_progress')
    rows_done = 0
    fetch_errors = []

    for pen, sid in pen_to_studentid.items():
        gresp = session.get(f"{BASE_URL}/p0/api/cy/students/{sid}", headers=HEADERS)
        if gresp.status_code != 200:
            fetch_errors.append(f"PEN {pen}: HTTP {gresp.status_code}")
            continue
        s   = gresp.json().get("data", {})
        row = rows_done + 2

        for ci, (hdr, api_field, locked, dmap_key) in enumerate(COLUMNS, 1):
            val  = _api_to_label(api_field, dmap_key, s)
            cell = ws.cell(row, ci, val)
            cell.border     = BORDER
            cell.font       = FONT_LOCK if locked else FONT_NORM
            cell.fill       = FILL_LOCKED if locked else FILL_WHITE
            cell.alignment  = ALIGN_CTR if ci <= 6 else ALIGN_LEFT

        rows_done += 1
        if rows_done % 5 == 0 or rows_done == len(pen_to_studentid):
            _pct = int(rows_done / len(pen_to_studentid) * 100)
            from IPython.display import update_display
            update_display(_HTML(f"""
            <div style='background:#161b22;border:1px solid #1f6feb;border-radius:8px;
                        padding:12px 18px;font-family:Inter,sans-serif;margin:4px 0;'>
              <div style='color:#c9d1d9;font-size:0.85rem;margin-bottom:8px;'>
                Fetching <strong style='color:#C9A84C'>{len(pen_to_studentid)}</strong> student profiles ...
              </div>
              <div style='background:#0d1117;border-radius:4px;height:6px;overflow:hidden;'>
                <div style='height:6px;width:{_pct}%;background:linear-gradient(90deg,#C9A84C,#e8c96d);'></div>
              </div>
              <div style='color:#8b949e;font-size:0.75rem;margin-top:6px;'>{rows_done} / {len(pen_to_studentid)} done ({_pct}%)</div>
            </div>"""), display_id='es_progress')
        time.sleep(0.05)

    last_row = rows_done + 1
    def dv_range(col_letter): return f"{col_letter}2:{col_letter}{last_row}"
    hdr_to_col = {col[0]: get_column_letter(i+1) for i, col in enumerate(COLUMNS)}

    def add_inline_dv(header, options):
        dv = DataValidation(
            type="list",
            formula1='"' + ",".join(options) + '"',
            allow_blank=True
        )
        dv.error = "Select a value from the dropdown."
        dv.errorTitle = "Invalid value"
        dv.showErrorMessage = True
        dv.sqref = dv_range(hdr_to_col[header])
        ws.add_data_validation(dv)

    def add_ref_dv(header, ref_range):
        dv = DataValidation(type="list", formula1=ref_range, allow_blank=True)
        dv.error = "Select a value from the dropdown."
        dv.errorTitle = "Invalid value"
        dv.showErrorMessage = True
        dv.sqref = dv_range(hdr_to_col[header])
        ws.add_data_validation(dv)

    add_inline_dv("Blood Group",             list(BLOOD_GROUP.keys()))
    add_inline_dv("Social Category",         list(SOCIAL_CATEGORY.keys()))
    add_inline_dv("Minority Group",          list(MINORITY_GROUP.keys()))
    add_inline_dv("BPL Beneficiary",         list(YES_NO.keys()))
    add_inline_dv("AAY Beneficiary",         list(YES_NO.keys()))
    add_inline_dv("EWS / Disadvantaged",     list(YES_NO.keys()))
    add_inline_dv("CWSN",                    list(YES_NO.keys()))
    add_inline_dv("Disability Certificate",  list(DISABILITY_CERTI.keys()))
    add_inline_dv("Indian National",         list(YES_NO.keys()))
    add_inline_dv("Out-of-School Child",     list(YES_NO.keys()))
    add_inline_dv("When Mainstreamed",       list(OOSC_MAINSTREAMED.keys()))
    add_ref_dv("Mother Tongue",      mt_ref)
    add_ref_dv("Type of Impairment", imp_ref)


    # Branded footer row in Excel
    footer_row = rows_done + 3
    footer_cell = ws.cell(footer_row, 1,
        f'Generated by Eternal Student | eternalstudent@cnrb | School: {SCHOOL_ID}')
    footer_cell.font = Font(color='C9A84C', italic=True, size=9)

    wb.save(output_path)
    from google.colab import files
    files.download(output_path)
    _err_html = ''
    if fetch_errors:
        _err_html = f'<div style="color:#f85149;font-size:0.78rem;margin-top:6px;">'\
            + ''.join(f'<div>&#10007; {e}</div>' for e in fetch_errors) + '</div>'
    from IPython.display import update_display
    update_display(_HTML(f"""
    <div style='background:#161b22;border:1px solid #238636;border-radius:8px;
                padding:14px 20px;font-family:Inter,sans-serif;'>
      <div style='display:flex;align-items:center;gap:10px;margin-bottom:10px;'>
        <span style='color:#3fb950;font-size:1.3rem;'>&#10003;</span>
        <span style='color:#c9d1d9;font-weight:600;'>Excel generated successfully</span>
      </div>
      <div style='display:flex;gap:24px;'>
        <div style='color:#8b949e;font-size:0.82rem;'>
          <span style='color:#C9A84C;font-weight:600;'>{rows_done}</span> rows written
        </div>
        <div style='color:#8b949e;font-size:0.82rem;'>
          File: <span style='color:#58a6ff;font-family:monospace;'>{output_path}</span>
        </div>
      </div>
      {_err_html}
      <div style='margin-top:12px;padding:10px 14px;background:#0d1117;
                  border-radius:6px;color:#8b949e;font-size:0.8rem;'>
        Next: <strong style='color:#c9d1d9;'>Download / open the file, edit columns L to AD, save.</strong>
        Then run Step 8 to upload.
      </div>
    </div>"""), display_id='es_progress')

output_file = f"UDISE_GP_Update_{SCHOOL_ID}.xlsx"
generate_excel(output_file)


In [ ]:
#@title 📄 General Profile — Upload Excel { display-mode: "form" }
def pick_file():
    """File picker: Colab dialog, then tkinter, then typed path."""
    try:
        from google.colab import files
        print("Choose your filled Excel file ...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file selected.")
        fname = list(uploaded.keys())[0]
        return fname, uploaded[fname]
    except ImportError:
        pass

    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        path = filedialog.askopenfilename(
            title="Select filled UDISE GP Update Excel",
            filetypes=[("Excel files", "*.xlsx"), ("All files", "*.*")]
        )
        root.destroy()
        if path:
            with open(path, "rb") as f:
                return os.path.basename(path), f.read()
    except Exception:
        pass

    path = input("Enter full path to filled Excel file: ").strip().strip('"')
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, "rb") as f:
        return os.path.basename(path), f.read()

UPLOAD_NAME, UPLOAD_BYTES = pick_file()
from IPython.display import HTML, display
display(HTML(f"""
<div style="background:#161b22;border:1px solid #1f6feb;border-radius:8px;
            padding:12px 18px;font-family:Inter,sans-serif;font-size:0.85rem;
            display:flex;align-items:center;gap:12px;">
  <span style="color:#58a6ff;font-size:1.3rem;">&#128190;</span>
  <div>
    <div style="color:#c9d1d9;font-weight:600;">{UPLOAD_NAME}</div>
    <div style="color:#8b949e;font-size:0.78rem;">{len(UPLOAD_BYTES):,} bytes &mdash; ready to process</div>
  </div>
</div>"""))


In [ ]:
#@title 📄 General Profile — Validate Excel { display-mode: "form" }

from IPython.display import HTML, display

def safe_str(v, empty=""):
    """Safe string -- returns empty if None/blank/nan."""
    if v is None or str(v).strip().lower() in ("", "nan", "none"):
        return empty
    return str(v).strip()

def safe_int(v, fallback=None):
    """Safe int."""
    try:
        return int(float(v))
    except:
        return fallback

def safe_mob(v):
    """Mobile: None when blank (API rejects empty string)."""
    s = safe_str(v)
    return None if s == "" else s

def safe_enum(v, dmap, field):
    s = safe_str(v)

    if s == "":
        return None

    if s in dmap:
        return dmap[s]

    raise ValueError(
        f"[{field}] Unknown value: {repr(s)}. "
        f"Valid: {list(dmap.keys())[:5]} ..."
    )

def safe_yn(v, field):
    return safe_enum(v, YES_NO, field)

def build_payload(row, original):

    """row: dict of {Excel header: value};
       original: full GET student record."""

    def ex(col, fallback=None):

        v = row.get(col)

        if v is None or str(v).strip().lower() in ("", "nan", "none"):
            return fallback

        return str(v).strip()

    p = {

        "classId"            : str(safe_int(original.get("classId"))),
        "sectionId"          : str(safe_int(original.get("sectionId"))),
        "studentId"          : str(safe_int(original.get("studentId"))),
        "schoolId"           : str(safe_int(original.get("schoolId"))),

        "uuidUpdateYN"       : 2,

        "gender"             : original.get("gender"),
        "dob"                : safe_str(original.get("dob")),

        "motherName"         : safe_str(original.get("motherName")),
        "fatherName"         : safe_str(original.get("fatherName")),
        "guardianName"       : safe_str(original.get("guardianName")),

        "uuid"               : "",
        "nameAsUuid"         : "",

        "studentCodeState"   : "",

        "certifiedCheckCount": 0,

        "ageCheckSkipped"    : safe_int(
                                    original.get("ageCheckSkipped"),
                                    2
                               ),
    }

    # ---------------- Basic Fields ----------------

    p["address"] = safe_str(
        ex("Address", original.get("address"))
    )

    p["pincode"] = safe_int(
        ex("Pincode", original.get("pincode"))
    )

    p["primaryMobile"] = safe_str(
        ex("Mobile No.", original.get("primaryMobile"))
    )

    p["secondaryMobile"] = safe_mob(
        ex(
            "Alternate Mobile No.",
            original.get("secondaryMobile")
        )
    )

    p["email"] = safe_str(
        ex("Contact Email", original.get("email"))
    )

    # ---------------- Dropdown Mappings ----------------

    mt = ex("Mother Tongue")

    p["motherTongue"] = (
        safe_enum(
            mt,
            MOTHER_TONGUE,
            "Mother Tongue"
        )
        if mt
        else original.get("motherTongue")
    )

    sc = ex("Social Category")

    p["socCatId"] = (
        safe_enum(
            sc,
            SOCIAL_CATEGORY,
            "Social Category"
        )
        if sc
        else original.get("socCatId")
    )

    mg = ex("Minority Group")

    p["minorityId"] = (
        safe_enum(
            mg,
            MINORITY_GROUP,
            "Minority Group"
        )
        if mg
        else original.get("minorityId")
    )

    # ---------------- YES / NO Fields ----------------

    for col, key, orig_key in [

        ("BPL Beneficiary",     "isBplYN",  "isBplYN"),
        ("AAY Beneficiary",     "aayBplYN", "aayBplYN"),
        ("EWS / Disadvantaged", "ewsYN",    "ewsYN"),
        ("CWSN",                "cwsnYN",   "cwsnYN"),
        ("Indian National",     "natIndYN", "natIndYN"),
        ("Out-of-School Child", "ooscYN",   "ooscYN"),

    ]:

        raw = ex(col)

        p[key] = (
            safe_yn(raw, col)
            if raw
            else original.get(orig_key, 2)
        )

    # ==================================================
    # CWSN / Impairment Logic
    # ==================================================

    it = ex("Type of Impairment")

    if p["cwsnYN"] == 1:

        # CWSN student → impairment required

        if it:

            mapped_it = safe_enum(
                it,
                IMPAIRMENT_TYPE,
                "Type of Impairment"
            )

            # IMPORTANT:
            # Backend expects ARRAY format

            p["impairmentType"] = [mapped_it]

        else:

            orig_it = original.get("impairmentType")

            if isinstance(orig_it, list):

                p["impairmentType"] = orig_it

            elif orig_it:

                p["impairmentType"] = [orig_it]

            else:

                raise ValueError(
                    "[Type of Impairment] "
                    "Required for CWSN students"
                )

        # Disability Certificate

        dc = ex("Disability Certificate")

        p["disabilityCerti"] = (

            safe_enum(
                dc,
                DISABILITY_CERTI,
                "Disability Certificate"
            )

            if dc

            else original.get(
                "disabilityCerti",
                2
            )
        )

        # Disability Percentage

        dp = safe_int(
            ex("Disability % (0-100)"),
            0
        )

        p["impairmentPercent"] = (
            ""
            if dp == 0
            else str(dp)
        )

    else:

        # Non-CWSN students

        p["impairmentType"] = []

        p["disabilityCerti"] = 9

        p["impairmentPercent"] = ""

    # ---------------- OOSC ----------------

    om = ex("When Mainstreamed")

    p["ooscMainstreamedYN"] = (

        safe_enum(
            om,
            OOSC_MAINSTREAMED,
            "When Mainstreamed"
        )

        if om

        else str(
            original.get(
                "ooscMainstreamedYN",
                "9"
            )
        )
    )

    # ---------------- Blood Group ----------------

    bg = ex("Blood Group")

    p["bloodGroup"] = (

        safe_enum(
            bg,
            BLOOD_GROUP,
            "Blood Group"
        )

        if bg

        else str(
            original.get(
                "bloodGroup",
                "9"
            )
        )
    )

    return p

# ==================================================
# LOAD EXCEL
# ==================================================

df = pd.read_excel(
    io.BytesIO(UPLOAD_BYTES),
    header=0,
    dtype=str
)

df.columns = [
    str(c).strip()
    for c in df.columns
]

df = df.apply(
    lambda col: col.map(
        lambda x: x.strip()
        if isinstance(x, str)
        else x
    )
)

# Remove fully blank rows

df = df.dropna(how="all")

# ==================================================
# COUNT UPDATABLE COLUMNS
# ==================================================

_updatable_cols = [

    c for c in df.columns

    if c not in (

        "Class",
        "Section",
        "DOB (DD/MM/YYYY)",
        "PEN",
        "Student Name",
        "Gender",
        "Mother Name",
        "Father Name",
        "Guardian Name",
        "Aadhaar No.",
        "Name as per Aadhaar"

    )
]

# ==================================================
# SUCCESS DASHBOARD
# ==================================================

display(HTML(f"""

<div style="background:#161b22;
            border:1px solid #238636;
            border-radius:8px;
            padding:14px 20px;
            font-family:Inter,sans-serif;">

  <div style="color:#3fb950;
              font-weight:600;
              margin-bottom:10px;">

      &#10003; Excel loaded and validated

  </div>

  <div style="display:flex;gap:28px;">

    <div style="text-align:center;">

      <div style="color:#C9A84C;
                  font-size:1.6rem;
                  font-weight:700;">

          {len(df)}

      </div>

      <div style="color:#8b949e;
                  font-size:0.75rem;">

          Student Rows

      </div>

    </div>

    <div style="width:1px;background:#30363d;"></div>

    <div style="text-align:center;">

      <div style="color:#58a6ff;
                  font-size:1.6rem;
                  font-weight:700;">

          {len(_updatable_cols)}

      </div>

      <div style="color:#8b949e;
                  font-size:0.75rem;">

          Updatable Columns

      </div>

    </div>

    <div style="width:1px;background:#30363d;"></div>

    <div style="display:flex;align-items:center;">

      <span style="color:#c9d1d9;
                   font-size:0.82rem;">

          Ready for Step 10

      </span>

    </div>

  </div>

</div>

"""))

In [ ]:
#@title 🚀 General Profile — Submit Updates { display-mode: "form" }
import random
results = []
errors  = []
_total  = len(df)

# Live dashboard
_dash = display(HTML(f"""
<div style="background:#161b22;border:1px solid #1f6feb;border-radius:8px;
            padding:14px 20px;font-family:Inter,sans-serif;">
  <div style="color:#c9d1d9;font-weight:600;margin-bottom:10px;">
    Updating <strong style="color:#C9A84C">{_total}</strong> students ...
  </div>
  <div id="es-submit-bar-bg" style="background:#0d1117;border-radius:4px;height:8px;overflow:hidden;margin-bottom:10px;">
    <div id="es-submit-bar" style="height:8px;width:0%;background:linear-gradient(90deg,#C9A84C,#3fb950);"></div>
  </div>
  <div id="es-submit-stats" style="display:flex;gap:24px;">
    <div style="color:#3fb950;font-size:0.82rem;">&#10003; Success: <strong>0</strong></div>
    <div style="color:#f85149;font-size:0.82rem;">&#10007; Failed: <strong>0</strong></div>
    <div style="color:#8b949e;font-size:0.82rem;">Skipped: <strong>0</strong></div>
  </div>
</div>"""), display_id="es_submit")

for idx, row in df.iterrows():
    pen = safe_str(row.get("PEN", "")).split(".")[0]
    if not pen:
        errors.append({"row": idx+2, "pen": "-", "name": "-", "error": "Missing PEN"})
        continue

    sid = pen_to_studentid.get(pen)
    if not sid:
        errors.append({"row": idx+2, "pen": pen, "name": "-",
                       "error": "PEN not found in school roster"})
        continue

    resp = session.get(f"{BASE_URL}/p0/api/cy/students/{sid}", headers=HEADERS)
    if resp.status_code != 200:
        errors.append({"row": idx+2, "pen": pen, "name": "-",
                       "error": f"Profile fetch failed: HTTP {resp.status_code}"})
        continue

    original = resp.json().get("data", {})
    name     = original.get("studentName", "")

    try:
        payload = build_payload(row.to_dict(), original)
    except ValueError as e:
        errors.append({"row": idx+2, "pen": pen, "name": name, "error": str(e)})
        continue

    post = session.post(f"{BASE_URL}/p0/api/cy/students/{sid}",
                        headers=HEADERS, json=payload)
    rj   = post.json()
    ok   = post.status_code == 200 and rj.get("status", False)

    if ok:
        results.append({"pen": pen, "name": name, "status": "SUCCESS",
                        "message": rj.get("message", "")})
    else:
        msg = rj.get("message", post.text[:120])
        results.append({"pen": pen, "name": name, "status": "FAILED", "message": msg})
        errors.append({"row": idx+2, "pen": pen, "name": name, "error": msg})

    # Update dashboard every 5 students
    _done    = len(results)
    _success = sum(1 for r in results if r["status"] == "SUCCESS")
    _failed  = _done - _success
    _skipped = len(errors) - _failed
    _pct     = int(_done / _total * 100) if _total else 100

    if _done % 5 == 0 or _done == _total:
        from IPython.display import update_display
        update_display(HTML(f"""
        <div style="background:#161b22;border:1px solid #1f6feb;border-radius:8px;
                    padding:14px 20px;font-family:Inter,sans-serif;">
          <div style="color:#c9d1d9;font-weight:600;margin-bottom:10px;">
            Updating <strong style="color:#C9A84C">{_total}</strong> students ...
          </div>
          <div style="background:#0d1117;border-radius:4px;height:8px;overflow:hidden;margin-bottom:10px;">
            <div style="height:8px;width:{_pct}%;background:linear-gradient(90deg,#C9A84C,#3fb950);"></div>
          </div>
          <div style="display:flex;gap:24px;">
            <div style="color:#3fb950;font-size:0.82rem;">&#10003; Success: <strong>{_success}</strong></div>
            <div style="color:#f85149;font-size:0.82rem;">&#10007; Failed: <strong>{_failed}</strong></div>
            <div style="color:#8b949e;font-size:0.82rem;">Skipped: <strong>{_skipped}</strong></div>
            <div style="color:#8b949e;font-size:0.82rem;">{_pct}% complete</div>
          </div>
        </div>"""), display_id="es_submit")

    time.sleep(random.uniform(0.4, 1.2))

# Final summary card
_success = sum(1 for r in results if r["status"] == "SUCCESS")
_failed  = len(results) - _success
_skipped = len([e for e in errors if e.get("pen") not in {r["pen"] for r in results}])
_err_rows = [e for e in errors if e.get("error")]
_err_html = "".join(
    f'<div style="padding:4px 0;border-bottom:1px solid #21262d;font-size:0.78rem;">&#10007; Row {e["row"]} | PEN {e["pen"]} | {e["name"]} | {e["error"]}</div>'
    for e in _err_rows
)

from IPython.display import update_display
update_display(HTML(f"""
<div style="background:#161b22;border:1px solid #238636;border-radius:8px;
            padding:20px 24px;font-family:Inter,sans-serif;">
  <div style="font-family:Cinzel,Georgia,serif;color:#C9A84C;font-size:0.75rem;
              letter-spacing:2px;margin-bottom:12px;text-transform:uppercase;">Eternal Student</div>
  <div style="color:#c9d1d9;font-size:1rem;font-weight:600;margin-bottom:16px;">
    General Profile Update &mdash; Complete
  </div>
  <div style="display:flex;gap:0;border:1px solid #30363d;border-radius:8px;overflow:hidden;margin-bottom:16px;">
    <div style="flex:1;padding:14px;text-align:center;border-right:1px solid #30363d;">
      <div style="color:#C9A84C;font-size:2rem;font-weight:700;">{len(df)}</div>
      <div style="color:#8b949e;font-size:0.75rem;margin-top:2px;">Total</div>
    </div>
    <div style="flex:1;padding:14px;text-align:center;border-right:1px solid #30363d;background:#0d2a16;">
      <div style="color:#3fb950;font-size:2rem;font-weight:700;">{_success}</div>
      <div style="color:#8b949e;font-size:0.75rem;margin-top:2px;">Success</div>
    </div>
    <div style="flex:1;padding:14px;text-align:center;border-right:1px solid #30363d;background:#2d1212;">
      <div style="color:#f85149;font-size:2rem;font-weight:700;">{_failed}</div>
      <div style="color:#8b949e;font-size:0.75rem;margin-top:2px;">Failed</div>
    </div>
    <div style="flex:1;padding:14px;text-align:center;">
      <div style="color:#8b949e;font-size:2rem;font-weight:700;">{_skipped}</div>
      <div style="color:#8b949e;font-size:0.75rem;margin-top:2px;">Skipped</div>
    </div>
  </div>
  {"<div style=\"background:#21262d;border-radius:6px;padding:10px 14px;max-height:180px;overflow-y:auto;\">" + _err_html + "</div>" if _err_html else ""}
  <div style="margin-top:14px;padding:10px 14px;background:#0d1117;border-radius:6px;
              color:#8b949e;font-size:0.8rem;display:flex;align-items:center;gap:8px;">
    <span style="color:#C9A84C;">&#128190;</span>
    Result log saved: <span style="color:#58a6ff;font-family:monospace;">UDISE_GP_Result_{SCHOOL_ID}.xlsx</span>
  </div>
</div>"""), display_id="es_submit")

# Save result log
log_path = f"UDISE_GP_Result_{SCHOOL_ID}.xlsx"
_all_rows = results + [e for e in errors
                        if e.get("pen") not in {r["pen"] for r in results}]
pd.DataFrame(_all_rows).to_excel(log_path, index=False)

from google.colab import files

files.download(log_path)



# 🔎 PEN SEARCH WITH AADHAAR NO. AND DOB in Bulk
<div style=" padding:20px; border-radius:10px; background:#fff3cd; border:1px solid #ffecb5; font-family:Arial; ">  <h2>🔐 How to Get Cookie & Encrypt Key</h2>  <ol style="font-size:16px; line-height:1.8;">  <li> Open: <b>https://sdms.udiseplus.gov.in/g0/</b> and login properly. </li>  <li> Press: <b>F12</b> or <b>Right Click → Inspect</b> </li>  <li> Open: <b>Console</b> tab </li>  <li> Paste this code in Console and press ENTER: </li>  </ol>  </div>  copy(sessionStorage.getItem("encryptDecryptKey"))  <div style=" padding:20px; border-radius:10px; background:#e8f5e9; border:1px solid #4caf50; font-family:Arial; margin-top:10px; ">  <h3>✅ What Happens?</h3>  <p> The Encrypt Key will automatically copy to clipboard. </p>  <p> Now paste it into: <b>ENCRYPT_KEY</b> field in Colab. </p>  <hr>  <h3>🍪 How to Get Cookie Header</h3>  <ol style="font-size:16px; line-height:1.8;">  <li> Open: <b>Network</b> tab in Inspect </li>  <li> Search any student manually once </li>  <li> Click: <b>search/nationalId</b> request </li>  <li> Go to: <b>Headers</b> </li>  <li> Copy complete: <b>Cookie</b> header </li>  </ol>  </div>

In [ ]:
# ============================================================
# 🔎 UDISE+ PEN SEARCH MODULE
# ============================================================

#@title 🔎 Search Student PEN using Aadhaar + DOB

!pip install pycryptodome openpyxl tqdm -q

import base64
import pandas as pd
import requests
import time

from datetime import datetime as _dt
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad
from google.colab import files
from IPython.display import display, HTML
from tqdm import tqdm


# ============================================================
# 📌 USER INPUT
# ============================================================

COOKIE_HEADER = "JSESSIONID=app_82_7008~DDF0B13AFB7C576C6CF4AD1E82925AE1; XSRF-TOKEN=179417f4-cd6d-4ed9-b717-919e8c6106a6; NSC_tent.vejtfqmvt_nqy_Wtfswfs_TTM=ffffffff09ca4c1d45525d5f4f58455e445a4a423660" #@param {type:"string"}

ENCRYPT_KEY = "x4rKOCf9QCeNtb0wzRTd0WZD0sjjSOfCBkwYz7pku44=" #@param {type:"string"}

SEARCH_DELAY = 1 #@param {type:"slider", min:0, max:5, step:1}


# ============================================================
# 🔐 AES ENCRYPT FUNCTION
# ============================================================

def encrypt_aadhaar(aadhaar, encrypt_key):

    key = base64.b64decode(encrypt_key)

    cipher = AES.new(key, AES.MODE_ECB)

    encrypted = cipher.encrypt(
        pad(aadhaar.encode(), AES.block_size)
    )

    return base64.b64encode(encrypted).decode()


# ============================================================
# 📅 SMART DATE
# Handles every date format seen in Indian school Excel files:
#   DD/MM/YYYY  DD-MM-YYYY  DD.MM.YYYY
#   YYYY-MM-DD  YYYY/MM/DD
#   DD/MM/YY    DD-MM-YY    (2-digit year)
#   DD Mon YYYY  DD Month YYYY  Mon DD, YYYY
#   Excel serial numbers (e.g. 40035)
#   datetime / Timestamp objects from pandas
# Returns DD/MM/YYYY string, or "" if unparseable.
# ============================================================

_DATE_FORMATS = [
    "%d/%m/%Y",   # 15/08/2010  ← most common in India
    "%d-%m-%Y",   # 15-08-2010
    "%d.%m.%Y",   # 15.08.2010
    "%Y-%m-%d",   # 2010-08-15  (ISO / software export)
    "%Y/%m/%d",   # 2010/08/15
    "%d/%m/%y",   # 15/08/10   (2-digit year)
    "%d-%m-%y",   # 15-08-10
    "%d %b %Y",   # 15 Aug 2010
    "%d %B %Y",   # 15 August 2010
    "%b %d, %Y",  # Aug 15, 2010
    "%B %d, %Y",  # August 15, 2010
    "%d-%b-%Y",   # 15-Aug-2010
]

def smart_date(value):
    """
    Parse any date value from an Excel cell.
    Returns DD/MM/YYYY string, or "" if unparseable.
    Never raises — safe to use in .apply().
    """

    # None / NaN
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""

    # Already a datetime / pandas Timestamp
    if hasattr(value, "strftime"):
        try:
            return value.strftime("%d/%m/%Y")
        except:
            return ""

    # Excel serial number stored as int or float (e.g. 40035.0)
    # Excel epoch: 1899-12-30 (Windows default)
    if isinstance(value, (int, float)):
        try:
            serial = int(value)
            if 1000 < serial < 80000:   # sanity range for valid dates
                epoch  = pd.Timestamp("1899-12-30")
                parsed = epoch + pd.Timedelta(days=serial)
                return parsed.strftime("%d/%m/%Y")
        except:
            pass
        return ""

    # String value — strip and normalise
    s = str(value).strip()

    if not s or s.lower() in ("nan", "nat", "none", ""):
        return ""

    # pandas auto-parse with dayfirst=True
    # (handles ambiguous DD/MM vs MM/DD correctly for Indian data)
    try:
        parsed = pd.to_datetime(s, dayfirst=True, errors="coerce")
        if not pd.isna(parsed):
            return parsed.strftime("%d/%m/%Y")
    except:
        pass

    # Explicit format fallback
    for fmt in _DATE_FORMATS:
        try:
            return _dt.strptime(s, fmt).strftime("%d/%m/%Y")
        except:
            continue

    return ""   # unparseable — returns empty, never crashes


def smart_year(value):
    """
    Extract the birth year from any date value.
    This API only needs the year for Aadhaar-based lookup.
    Returns 4-digit year string (e.g. "2010"), or "" if unparseable.
    """

    date_str = smart_date(value)

    if not date_str:
        # Last resort: if the raw value looks like a 4-digit year already
        s = str(value).strip()
        if len(s) == 4 and s.isdigit():
            return s
        return ""

    # date_str is DD/MM/YYYY — extract YYYY
    try:
        return date_str.split("/")[2]
    except:
        return ""


# ============================================================
# 🍪 PARSE COOKIES
# ============================================================

cookies = {}

for item in COOKIE_HEADER.split(";"):

    if "=" in item:

        key, value = item.strip().split("=", 1)

        cookies[key] = value


# ============================================================
# 🌐 REQUEST HEADERS
# ============================================================

HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Origin": "https://sdms.udiseplus.gov.in",
    "Referer": "https://sdms.udiseplus.gov.in/g0/",
    "User-Agent": "Mozilla/5.0",
    "X-XSRF-TOKEN": cookies.get("XSRF-TOKEN", "")
}


# ============================================================
# 🔗 API URL
# ============================================================

API_URL = (
    "https://sdms.udiseplus.gov.in/"
    "p0/api/students/search/nationalId"
)


# ============================================================
# 📤 UPLOAD EXCEL
# ============================================================

print("📤 Upload Aadhaar Excel File")

uploaded = files.upload()

input_file = list(uploaded.keys())[0]

# Load with dtype=str so pandas never silently converts dates
# to NaT during read. We handle all date parsing ourselves.
df = pd.read_excel(input_file, dtype=str)

# Re-read with native types so smart_date() gets original
# datetime objects / serial floats for date columns.
_df_raw = pd.read_excel(input_file)

display(HTML(
    "<h3 style='color:green'>✅ File Uploaded Successfully</h3>"
))

print("\nDetected Columns:")
print(df.columns.tolist())


# ============================================================
# 🧹 CLEAN AADHAAR / PEN ID COLUMNS
# (strip ".0" from numeric strings caused by dtype=str read)
# ============================================================

def _clean_id(x):
    if pd.isna(x) or str(x).strip().lower() in ("nan", "none", ""):
        return ""
    try:
        return str(int(float(str(x).strip())))
    except:
        return str(x).strip()

for col in df.columns:
    lower_col = str(col).lower().strip()
    if any(k in lower_col for k in ["aadhaar", "adhaar", "adhar", "uid", "pen"]):
        df[col] = df[col].apply(_clean_id)


# ============================================================
# 📅 APPLY SMART DATE TO DATE COLUMNS IN df
# Using raw re-read values for best accuracy.
# ============================================================

for col in df.columns:

    lower_col = str(col).lower().strip()

    if any(k in lower_col for k in ["dob", "birth", "admission"]):

        if col in _df_raw.columns:
            df[col] = _df_raw[col].apply(smart_date)
        else:
            df[col] = df[col].apply(smart_date)


# ============================================================
# 🔍 AUTO DETECT COLUMNS
# ============================================================

aadhaar_col = None
dob_col     = None

for col in df.columns:

    lower_col = str(col).lower().strip()

    # Aadhaar variations
    if any(k in lower_col for k in ["aadhaar", "adhaar", "adhar", "uid"]):
        aadhaar_col = col

    # DOB variations
    if any(k in lower_col for k in ["dob", "birth", "date of birth"]):
        dob_col = col


# ============================================================
# VALIDATION
# ============================================================

if aadhaar_col is None:

    raise Exception(
        f"Aadhaar column not detected.\n\n"
        f"Available Columns:\n{df.columns.tolist()}"
    )

if dob_col is None:

    raise Exception(
        f"DOB column not detected.\n\n"
        f"Available Columns:\n{df.columns.tolist()}"
    )


print(f"\n✅ Aadhaar Column : {aadhaar_col}")
print(f"✅ DOB Column      : {dob_col}")


# ============================================================
# 🚀 PROCESS STUDENTS
# ============================================================

# Add output columns
df["PEN"]      = ""
df["UDISE_DOB"] = ""
df["STATUS"]   = ""

success_count = 0
failed_count  = 0

print("\n🔄 Searching PENs...\n")

for index, row in tqdm(df.iterrows(), total=len(df)):

    try:

        # ====================================================
        # CLEAN AADHAAR
        # Already cleaned above but re-clean inside loop for
        # safety in case of stray spaces or formatting.
        # ====================================================

        aadhaar = str(row[aadhaar_col]).strip()

        # Remove spaces and keep digits only
        aadhaar = ''.join(filter(str.isdigit, aadhaar))

        # Skip empty Aadhaar
        if aadhaar == "":

            df.at[index, "STATUS"] = "EMPTY AADHAAR"

            continue

        # ====================================================
        # EXTRACT BIRTH YEAR FOR API
        # API only needs the year, not full DOB.
        # smart_year() handles all date formats safely.
        # ====================================================

        # Use raw value from _df_raw for best date parsing
        raw_dob = (
            _df_raw.at[index, dob_col]
            if dob_col in _df_raw.columns
            else row[dob_col]
        )

        dob = smart_year(raw_dob)

        # If year extraction failed, warn and skip
        if dob == "":

            df.at[index, "STATUS"] = "DOB EMPTY / INVALID"

            failed_count += 1

            continue

        # ====================================================
        # ENCRYPT AADHAAR
        # ====================================================

        encrypted_uuid = encrypt_aadhaar(
            aadhaar,
            ENCRYPT_KEY
        )

        payload = {
            "dob":  dob,
            "uuid": encrypted_uuid
        }

        # ====================================================
        # API REQUEST
        # ====================================================

        response = requests.post(
            API_URL,
            headers=HEADERS,
            cookies=cookies,
            json=payload,
            timeout=30
        )

        data = response.json()

        # ====================================================
        # SUCCESS
        # ====================================================

        if data.get("status") and data.get("data"):

            student = data["data"][0]

            pen = student.get("nationalId", "")

            # Save the exact DOB string returned by the API
            # (e.g. "09/08/2015") without any reformatting.
            # This is the authoritative DOB from UDISE — use
            # it directly in the import step.
            response_dob = str(
                student.get("dob", "")
            ).strip()

            df.at[index, "PEN"]      = pen
            df.at[index, "UDISE_DOB"] = response_dob
            df.at[index, "STATUS"]   = "FOUND"

            success_count += 1

        else:

            # Capture API message if available
            api_message = data.get("message", "Not found")

            df.at[index, "STATUS"] = f"NOT FOUND : {api_message}"

            failed_count += 1

        time.sleep(SEARCH_DELAY)

    except Exception as e:

        failed_count += 1

        df.at[index, "STATUS"] = f"ERROR : {e}"

        print(f"Error Row {index + 1}: {e}")


# ============================================================
# 💾 SAVE UPDATED FILE
# NOTE: UDISE_DOB is intentionally NOT reformatted here —
# it already contains the exact DD/MM/YYYY from the API.
# The other date columns were already normalised above
# before the loop, so no second pass is needed.
# ============================================================

OUTPUT_FILE = "Updated_" + input_file

df.to_excel(OUTPUT_FILE, index=False)


# ============================================================
# 📊 SUMMARY
# ============================================================

print("\n===================================")
print("✅ PROCESS COMPLETED")
print("===================================")

print(f"✅ Success : {success_count}")
print(f"❌ Failed  : {failed_count}")

display(HTML(f'''
<div style="
padding:15px;
border-radius:10px;
background:#e8f5e9;
border:1px solid #4caf50;
">

<h2>✅ PEN Search Completed</h2>

<p><b>Success:</b> {success_count}</p>
<p><b>Failed:</b> {failed_count}</p>

</div>
'''))


# ============================================================
# 📥 DOWNLOAD UPDATED FILE
# ============================================================

files.download(OUTPUT_FILE)

#  UDISE+ STUDENTS IMPORT in BULK

In [ ]:
# ============================================================
# 🎓 UDISE+ BULK STUDENT IMPORT SYSTEM
# ============================================================

!pip install openpyxl -q

import time
import requests
import pandas as pd
from google.colab import files
from IPython.display import display, HTML, clear_output
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

# ============================================================
# ⚠️ IMPORTANT NOTICE
# ============================================================

display(HTML("""
<div style="padding:18px;border-radius:12px;background:#fff8e1;
border:1px solid #ffcc00;font-family:Arial;">
<h2>⚠️ Important Notice</h2>
Please verify: PEN Number, Date of Birth, Class &amp; Section, Admission Date.
Incorrect data may import students incorrectly.
</div>
"""))

# ============================================================
# 🏫 CHECK SCHOOL
# ============================================================

try:
    SCHOOL_ID
except:
    raise Exception("❌ Run Detect School section first.")

# ============================================================
# 📌 USER INPUT
# ============================================================

COOKIE_HEADER = "JSESSIONID=app_82_7008~DDF0B13AFB7C576C6CF4AD1E82925AE1; XSRF-TOKEN=d485bb19-d507-447c-a25c-3bba357ec072; NSC_tent.vejtfqmvt_nqy_Wtfswfs_TTM=ffffffff09ca4c1d45525d5f4f58455e445a4a423660" #@param {type:"string"}
SEARCH_DELAY = 1 #@param {type:"slider", min:0, max:5, step:1}

# ============================================================
# 🍪 PARSE COOKIES
# ============================================================

cookies = {}
for item in COOKIE_HEADER.split(";"):
    if "=" in item:
        key, value = item.strip().split("=", 1)
        cookies[key] = value

# ============================================================
# 🌐 HEADERS
# ============================================================

HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Origin": "https://sdms.udiseplus.gov.in",
    "Referer": "https://sdms.udiseplus.gov.in/g0/",
    "User-Agent": "Mozilla/5.0",
    "X-XSRF-TOKEN": cookies.get("XSRF-TOKEN", "")
}

# ============================================================
# 🔗 API URLS
# ============================================================

SEARCH_API = f"https://sdms.udiseplus.gov.in/p0/api/students/import/search/{SCHOOL_ID}"
IMPORT_API = "https://sdms.udiseplus.gov.in/p0/api/students/import/submit/"

# ============================================================
# 📚 CLASS MAP & SECTION MAP
# ============================================================

CLASS_MAP = {
    "1":1,"2":2,"3":3,"4":4,"5":5,"6":6,
    "7":7,"8":8,"9":9,"10":10,"11":11,"12":12,
    "I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,
    "VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12
}

SECTION_MAP = {
    "A":1,"B":2,"C":3,"D":4,
    "E":5,"F":6,"G":7,"H":8
}

# ============================================================
# 📅 SMART DATE — handles every format schools use pan-India
# ============================================================

# All formats seen across Indian school Excel files
DATE_FORMATS = [
    "%d/%m/%Y",   # 15/08/2010  ← most common
    "%d-%m-%Y",   # 15-08-2010
    "%d.%m.%Y",   # 15.08.2010
    "%Y-%m-%d",   # 2010-08-15  (ISO, some software exports)
    "%Y/%m/%d",   # 2010/08/15
    "%d/%m/%y",   # 15/08/10   (2-digit year)
    "%d-%m-%y",   # 15-08-10
    "%d %b %Y",   # 15 Aug 2010
    "%d %B %Y",   # 15 August 2010
    "%b %d, %Y",  # Aug 15, 2010
    "%B %d, %Y",  # August 15, 2010
    "%d-%b-%Y",   # 15-Aug-2010
]

def smart_date(value):
    """
    Robustly parse any date value from an Excel cell and return
    DD/MM/YYYY string. Returns empty string if unparseable.
    Handles: datetime objects, Excel serial floats, strings in
    any of the common Indian school formats listed above.
    """

    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""

    # Already a datetime / Timestamp from pandas
    if hasattr(value, "strftime"):
        try:
            return value.strftime("%d/%m/%Y")
        except:
            return ""

    # Excel serial number stored as float/int (e.g. 40035.0)
    # Excel epoch: 1899-12-30 (Windows) / 1904-01-01 (Mac)
    if isinstance(value, (int, float)):
        try:
            serial = int(value)
            if 1000 < serial < 80000:          # sanity range for valid dates
                epoch = pd.Timestamp("1899-12-30")
                parsed = epoch + pd.Timedelta(days=serial)
                return parsed.strftime("%d/%m/%Y")
        except:
            pass
        return ""

    # String value — clean it first
    s = str(value).strip()

    if not s or s.lower() in ("nan", "nat", "none", ""):
        return ""

    # Try pandas auto-parse with dayfirst=True (covers ambiguous DD/MM vs MM/DD)
    try:
        parsed = pd.to_datetime(s, dayfirst=True, errors="coerce")
        if not pd.isna(parsed):
            return parsed.strftime("%d/%m/%Y")
    except:
        pass

    # Try each explicit format
    from datetime import datetime
    for fmt in DATE_FORMATS:
        try:
            parsed = datetime.strptime(s, fmt)
            return parsed.strftime("%d/%m/%Y")
        except:
            continue

    return ""   # give up — log as empty, don't crash

# ============================================================
# 🖥️ LIVE TERMINAL
# ============================================================

terminal_logs = []

def terminal(message):
    terminal_logs.append(message)
    if len(terminal_logs) > 18:
        terminal_logs.pop(0)
    clear_output(wait=True)
    html = f"""
    <div style="background:#0d1117;color:#58ff8a;padding:18px;
    border-radius:12px;font-family:monospace;border:2px solid #30363d;
    height:420px;overflow-y:auto;box-shadow:0 0 12px rgba(0,0,0,0.35);">
    <div style="color:#79c0ff;font-size:20px;margin-bottom:12px;font-weight:bold;">
    🎓 UDISE+ BULK IMPORT SYSTEM</div>
    {'<br>'.join(terminal_logs)}
    </div>
    """
    display(HTML(html))

# ============================================================
# 📤 UPLOAD EXCEL
# ============================================================

print("📤 Upload Excel File")
uploaded = files.upload()
input_file = list(uploaded.keys())[0]

# Read without any date parsing — let smart_date() handle everything
df = pd.read_excel(input_file, dtype=str)

# ============================================================
# 🧹 CLEAN PEN & AADHAAR  (strip .0 from numeric strings)
# ============================================================

for col in ["PEN", "ADHAAR NO."]:
    if col in df.columns:
        def clean_id(x):
            if pd.isna(x) or str(x).strip().lower() in ("nan","none",""):
                return ""
            try:
                return str(int(float(str(x).strip())))
            except:
                return str(x).strip()
        df[col] = df[col].apply(clean_id)

# ============================================================
# 📅 APPLY SMART DATE TO DATE COLUMNS
# ============================================================

date_columns = ["DATE OF BIRTH", "DATE OF ADMISSION"]

# Re-read raw values for date columns so we get original Excel values
df_raw = pd.read_excel(input_file)   # re-read with native types for dates

for col in date_columns:
    if col in df.columns and col in df_raw.columns:
        df[col] = df_raw[col].apply(smart_date)
    elif col in df.columns:
        df[col] = df[col].apply(smart_date)

# ============================================================
# 📄 REQUIRED COLUMNS CHECK
# ============================================================

required_columns = ["PEN","CLASS","SEC","DATE OF BIRTH","DATE OF ADMISSION"]
missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise Exception(f"❌ Missing Columns: {missing_columns}")

# ============================================================
# ➕ OUTPUT COLUMNS
# ============================================================

for col in ["IMPORT_STATUS","REMARK","ELIGIBLE_CLASSES","PREVIOUS_SCHOOL","PREVIOUS_UDISE"]:
    if col not in df.columns:
        df[col] = ""

# ============================================================
# 📊 COUNTERS
# ============================================================

success_count = 0
failed_count  = 0
skipped_count = 0

# ============================================================
# 🚀 START
# ============================================================

terminal(f"🏫 SCHOOL ID : {SCHOOL_ID}")
terminal(f"📄 TOTAL STUDENTS : {len(df)}")
terminal("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

# ============================================================
# 🔄 PROCESS STUDENTS
# ============================================================

for index, row in df.iterrows():

    try:

        pen            = str(row["PEN"]).strip()
        dob            = str(row["DATE OF BIRTH"]).strip()
        class_name     = str(row["CLASS"]).strip().upper().replace(".", "")
        section_name   = str(row["SEC"]).strip().upper()
        admission_date = str(row["DATE OF ADMISSION"]).strip()

        terminal(f"🔍 [{index+1}/{len(df)}] PEN: {pen} | DOB: {dob} | ADM: {admission_date}")

        # ── Empty PEN ──────────────────────────────────────
        if not pen:
            terminal("⚠️ PEN EMPTY")
            df.at[index, "IMPORT_STATUS"] = "SKIPPED"
            df.at[index, "REMARK"]        = "PEN EMPTY"
            skipped_count += 1
            continue

        # ── Warn if DOB empty but still try ───────────────
        if not dob:
            terminal("⚠️ DOB EMPTY — trying search with PEN only")

        # ── Search payload ─────────────────────────────────
        search_payload = {
            "searchType":     1,
            "studentCodeNat": pen,
            "dob":            dob
        }

        search_response = requests.post(
            SEARCH_API, headers=HEADERS,
            cookies=cookies, json=search_payload, timeout=30
        )
        search_data = search_response.json()

        # ── Student not found ──────────────────────────────
        if not search_data.get("status") or not search_data.get("data"):
            terminal("❌ STUDENT NOT FOUND")
            df.at[index, "IMPORT_STATUS"] = "NOT FOUND"
            df.at[index, "REMARK"]        = (
                "Student not found — check PEN / DOB in Excel"
                if dob else
                "Student not found — DOB was empty, verify PEN"
            )
            failed_count += 1
            continue

        # ── Student details ────────────────────────────────
        student      = search_data["data"][0]
        student_name = student.get("studentName", "")
        status_desc  = str(student.get("statusDesc", "")).strip()

        school           = student.get("school")    or {}
        school_py        = student.get("schoolPY")  or {}
        current_school   = school.get("schoolName", "")
        previous_school  = school_py.get("schoolName", "")
        previous_udise   = school_py.get("udiseSchCode", "")

        df.at[index, "PREVIOUS_SCHOOL"] = previous_school
        df.at[index, "PREVIOUS_UDISE"]  = previous_udise

        terminal(f"✅ FOUND : {student_name}")

        # ── Active in another school ───────────────────────
        if status_desc == "ACTIVE":
            terminal(f"🏫 ACTIVE IN : {current_school}")
            df.at[index, "IMPORT_STATUS"] = current_school
            df.at[index, "REMARK"]        = "ACTIVE"
            failed_count += 1
            continue

        # ── Not eligible ───────────────────────────────────
        if "Dropbox" not in status_desc:
            terminal(f"❌ STATUS : {status_desc}")
            df.at[index, "IMPORT_STATUS"] = status_desc
            df.at[index, "REMARK"]        = "NOT ELIGIBLE"
            failed_count += 1
            continue

        terminal("📥 ELIGIBLE FOR IMPORT")

        # ── Class / section validation ─────────────────────
        if class_name not in CLASS_MAP:
            terminal(f"❌ INVALID CLASS : '{class_name}'")
            df.at[index, "IMPORT_STATUS"] = "FAILED"
            df.at[index, "REMARK"]        = f"INVALID CLASS : {class_name}"
            failed_count += 1
            continue

        if section_name not in SECTION_MAP:
            terminal(f"❌ INVALID SECTION : '{section_name}'")
            df.at[index, "IMPORT_STATUS"] = "FAILED"
            df.at[index, "REMARK"]        = f"INVALID SECTION : {section_name}"
            failed_count += 1
            continue

        class_id   = CLASS_MAP[class_name]
        section_id = SECTION_MAP[section_name]

        # ── Eligible class check ───────────────────────────
        eligible_classes    = student.get("eligibleClasses", []) or []
        eligible_class_ids  = [int(ec.get("id", 0)) for ec in eligible_classes]
        eligible_class_names = ", ".join([ec.get("value", "") for ec in eligible_classes])

        df.at[index, "ELIGIBLE_CLASSES"] = eligible_class_names or "NONE"

        if eligible_class_ids and class_id not in eligible_class_ids:
            terminal(f"⚠️ CLASS MISMATCH | YOURS: {class_name} | ELIGIBLE: {eligible_class_names}")
            df.at[index, "IMPORT_STATUS"] = "FAILED"
            df.at[index, "REMARK"] = (
                f"CLASS MISMATCH : You entered Class {class_name} "
                f"but student is eligible for Class {eligible_class_names} only"
            )
            failed_count += 1
            continue

        # ── Admission date warning ─────────────────────────
        if not admission_date:
            terminal("⚠️ ADMISSION DATE EMPTY — import may fail")

        # ── Import payload ─────────────────────────────────
        enrolment = student.get("enrolmentDetails", {}) or {}

        import_payload = {
            "studentId":            student.get("studentId"),
            "schoolId":             SCHOOL_ID,
            "classId":              class_id,
            "progressionStatusCy":  1,
            "marksCy":              enrolment.get("marksPy", 0),
            "attendanceCy":         enrolment.get("attendancePy", 0),
            "sectionId":            str(section_id),
            "admissionDate":        admission_date
        }

        terminal(f"📦 classId={class_id} sectionId={section_id} admDate={admission_date}")

        import_response = requests.post(
            IMPORT_API, headers=HEADERS,
            cookies=cookies, json=import_payload, timeout=30
        )
        import_data = import_response.json()

        # ── Import result ──────────────────────────────────
        if import_data.get("status"):
            terminal(f"🎉 IMPORTED : {student_name}")
            df.at[index, "IMPORT_STATUS"] = "IMPORTED"
            df.at[index, "REMARK"]        = "SUCCESS"
            success_count += 1
        else:
            api_message = import_data.get("message", "Import failed — no message from API")
            terminal(f"❌ IMPORT FAILED : {api_message}")
            df.at[index, "IMPORT_STATUS"] = "FAILED"
            df.at[index, "REMARK"]        = api_message
            failed_count += 1

        time.sleep(SEARCH_DELAY)

    except Exception as e:
        terminal(f"❌ ERROR : {e}")
        df.at[index, "IMPORT_STATUS"] = "ERROR"
        df.at[index, "REMARK"]        = str(e)
        failed_count += 1

# ============================================================
# 💾 SAVE & STYLE OUTPUT EXCEL
# ============================================================

OUTPUT_FILE = "Imported_" + input_file
df.to_excel(OUTPUT_FILE, index=False)

wb = load_workbook(OUTPUT_FILE)
ws = wb.active
ws.freeze_panes = "A2"

header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font

for column_cells in ws.columns:
    length = max(
        len(str(cell.value)) if cell.value else 0
        for cell in column_cells
    )
    ws.column_dimensions[get_column_letter(column_cells[0].column)].width = length + 4

wb.save(OUTPUT_FILE)

# ============================================================
# 📊 FINAL SUMMARY
# ============================================================

terminal("━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
terminal(f"✅ IMPORTED  : {success_count}")
terminal(f"❌ FAILED    : {failed_count}")
terminal(f"⏭️  SKIPPED   : {skipped_count}")
terminal("🎉 PROCESS COMPLETED")

files.download(OUTPUT_FILE)

# Made with
###  ❤️ by Eternal Student

If this tool saved your time and effort,
consider supporting with Tea ☕ or Coffee ☕

**UPI:** `eternalstudent@cnrb`